In [1]:
!pip install --upgrade google-cloud-bigquery google-cloud-aiplatform vertexai pandas

INFO: pip is looking at multiple versions of google-cloud-aiplatform to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-cloud-aiplatform to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 17.7 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: google-cloud-bigquery
    Found existing installation: google-cloud-bigquery 3.40.0
    Uninstalling google-cloud-bigquery-3.40.0:
      Successfully uninstalled google-cloud-bigquery-3.40.0
  Attempting uninstall: google-cloud-aiplatform━ 0/3 [google-cloud-bigquery]
    Found existing installation: google-cloud-aiplatfo

In [16]:
import vertexai
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "search-me-cs226"
REGION = "us-central1"

vertexai.init(project=PROJECT_ID, location=REGION)

print("Vertex AI initialized successfully")

Vertex AI initialized successfully


In [23]:
client = bigquery.Client()

query = """
SELECT repo_name, readme_text
FROM `search-me-cs226.searchme_dataset.cleaned_readme_subset_50k`
WHERE readme_text IS NOT NULL
LIMIT 55000
"""

df = client.query(query).to_dataframe()

print("Loaded rows: ", len(df))
df.head()

Loaded rows:  7364


,repo_name,readme_text
0,otalk/sdp-jingle-json,# SDP-Jingle-JSON\n**Convert SDP blobs to and ...
1,hapijs/topo,# topo\n\nTopological sorting with grouping su...
2,nativelibs4java/JavaCL,[![Maven Central](https://img.shields.io/maven...
3,openfl/swf,[![MIT License](https://img.shields.io/badge/l...
4,RcppCore/rcpp-gallery,\n## Rcpp Gallery\n\nThis is the repository fo...


In [26]:
from vertexai.language_models import TextEmbeddingModel
model = TextEmbeddingModel.from_pretrained("gemini-embedding-001")

def get_embedding(text):
    if not text:
        return None
    return model.get_embeddings([text])[0].values

df["embedding"] = df["readme_text"].apply(get_embedding)

print("Embeddings created")
len(df["embedding"][0])

/opt/conda/lib/python3.10/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Embeddings created


3072

In [27]:
table_id = "search-me-cs226.searchme_dataset.subset_50k_gemini_embeddings"

df_upload = df[["repo_name", "readme_text", "embedding"]]

job = client.load_table_from_dataframe(
    df_upload,
    table_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE"
    )
)

job.result()

print("Embeddings table created")

Embeddings table created


In [14]:
# Links used
# https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemini-embedding-001
# https://docs.cloud.google.com/vertex-ai/docs/workbench/instances/bigquery
